Imports and loading

In [9]:
import pandas as pd
import numpy as np

# The simple model we'll use to compare the strategies
from sklearn.linear_model import LogisticRegression

# A "dumb" model that never learns anything; use as a baseline to show why accuracy fails
from sklearn.dummy import DummyClassifier

# Metrics to judge the mode
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,    # PR_AUC
    confusion_matrix,
)
# SMOTE creates synthetic fraud samples; RandomUnderSample removes normal samples
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

load the data saved

In [10]:
# Read the training and test files we saved in stages 3
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

# The target files have one column calledd "Class"; we take it as a Series
y_train = pd.read_csv("../data/processed/y_train.csv")["Class"]
y_test  = pd.read_csv("../data/processed/y_test.csv")["Class"]

print("Train:", X_train.shape, "| Test:",X_test.shape)
print("Fraud in train:", y_train.sum(), "| Fraud in test:",y_test.sum())


Train: (226980, 30) | Test: (56746, 30)
Fraud in train: 378 | Fraud in test: 95


Why accuracy is misleading

In [11]:
# This model always predicts the most common class (0 = nomal),no matter the input 

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train,y_train)

# Predict on the tesst set
dummy_pred = dummy.predict(X_test)

# Accuracy looks amazing...
print("Accuracy:", round(accuracy_score(y_test,dummy_pred),4))
# ...but it catches ZERO fraud (recall = fraction of real frauds we found)
print("Recall: ", recall_score(y_test, dummy_pred))

Accuracy: 0.9983
Recall:  0.0


In [12]:
def evaluate_model(name, model, X_test, y_test):
    """Evaluate a trained model on the test set and return the scores as a dictionary."""

    # Predicted class (0 or 1) for each test transaction
    y_pred = model.predict(X_test)

    # Predicted probability of fraud (needed for PR-AUC); column 1 = probability of class 1
    y_prob = model.predict_proba(X_test)[:, 1]

    # Confusion matrix gives: true negatives, false positives, false negatives, true positives
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    # Collect everything in one dictionary
    return {
        "Model": name,
        "Precision": round(precision_score(y_test, y_pred), 3),
        "Recall": round(recall_score(y_test, y_pred), 3),
        "F1": round(f1_score(y_test, y_pred), 3),
        "PR-AUC": round(average_precision_score(y_test, y_prob), 3),
        "Frauds caught (TP)": tp,
        "Frauds missed (FN)": fn,
        "False alarms (FP)": fp,
    }

# We'll store each result here and compare at the end
results = []

Baseline (no imbalance handling)

In [13]:
# Plain Logistic Regression, trained on the imbalanced data as-is
# max_iter=1000 gives the algorithm enough steps to converge
baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train, y_train)

# Evaluate and store the result
results.append(evaluate_model("Baseline (no fix)", baseline, X_test, y_test))
pd.DataFrame(results)

,Model,Precision,Recall,F1,PR-AUC,Frauds caught (TP),Frauds missed (FN),False alarms (FP)
0,Baseline (no fix),0.862,0.589,0.7,0.695,56,39,9


Fix 1, class weights

This tells the model: “a mistake on a fraud case costs much more than a mistake on a normal case.” No data is changed.

In [15]:
# class_weight="balanced" automatically gives the rare class a much higher weight
weighted = LogisticRegression(max_iter=1000, class_weight="balanced")
weighted.fit(X_train, y_train)

results.append(evaluate_model("Class weights", weighted, X_test, y_test))
pd.DataFrame(results)

,Model,Precision,Recall,F1,PR-AUC,Frauds caught (TP),Frauds missed (FN),False alarms (FP)
0,Baseline (no fix),0.862,0.589,0.700,0.695,56,39,9
1,Class weights,0.056,0.874,0.106,0.672,83,12,1393


Fix 2, SMOTE (oversampling)

SMOTE creates new synthetic fraud examples by interpolating between existing real fraud cases, until fraud and normal are balanced.

In [16]:
# Create the SMOTE object; random_state makes the result repeatable
smote = SMOTE(random_state=42)

# fit_resample creates synthetic fraud rows. IMPORTANT: only applied to TRAINING data.
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Compare class counts before and after
print("Before SMOTE:")
print(y_train.value_counts())
print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

Before SMOTE:
Class
0    226602
1       378
Name: count, dtype: int64

After SMOTE:
Class
0    226602
1    226602
Name: count, dtype: int64


In [17]:
# Train on the balanced (SMOTE) training data
smote_model = LogisticRegression(max_iter=1000)
smote_model.fit(X_train_smote, y_train_smote)

# Evaluate on the ORIGINAL, untouched test set (never resample the test set)
results.append(evaluate_model("SMOTE", smote_model, X_test,y_test))
pd.DataFrame(results)

,Model,Precision,Recall,F1,PR-AUC,Frauds caught (TP),Frauds missed (FN),False alarms (FP)
0,Baseline (no fix),0.862,0.589,0.700,0.695,56,39,9
1,Class weights,0.056,0.874,0.106,0.672,83,12,1393
2,SMOTE,0.054,0.874,0.101,0.673,83,12,1463


Fix 3, random undersampling

In [18]:
# Randomly remove normal rows from the training data until both classes have equal size
under = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = under.fit_resample(X_train,y_train)

print("After undersampling:")
print(y_train_under.value_counts()) # both classes now have only ~378

After undersampling:
Class
0    378
1    378
Name: count, dtype: int64


In [19]:
# Train on the small balanced dataset
under_model = LogisticRegression(max_iter=1000)
under_model.fit(X_train_under, y_train_under)

results.append(evaluate_model("Undersampling", under_model, X_test, y_test))

In [20]:
# Put all results into one table, sorted by PR-AUC (our main metric)
comparison = pd.DataFrame(results).sort_values("PR-AUC", ascending=False)
comparison

,Model,Precision,Recall,F1,PR-AUC,Frauds caught (TP),Frauds missed (FN),False alarms (FP)
0,Baseline (no fix),0.862,0.589,0.700,0.695,56,39,9
2,SMOTE,0.054,0.874,0.101,0.673,83,12,1463
1,Class weights,0.056,0.874,0.106,0.672,83,12,1393
3,Undersampling,0.051,0.874,0.096,0.600,83,12,1552


The key lesson: there is a trade-off between precision and recall. Fixing imbalance raises recall (you catch more fraud) but usually lowers precision (you flag more innocent transactions). A real bank has to decide how many false alarms it can tolerate. In Stage 6 we’ll learn to tune the decision threshold to control this.